# Fine-tuning DistilBERT Indonesia
        **text › classification** · notebook `finetune_distilbert_id`

        **Kapan dipakai:** Akurasi tertinggi pada teks Indonesia bila ada waktu dan (idealnya) GPU. Mengungguli TF-IDF terutama pada tugas yang butuh pemahaman konteks.

        **Jebakan umum:** Di CPU hanya demo (subset kecil, 1 epoch): naikkan N_TRAIN/EPOCHS saat dipakai sungguhan. Sensitif learning rate (2e-5 sampai 5e-5). Selalu bandingkan dengan baseline TF-IDF.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

> **Demo CPU**: subset kecil dan 1 epoch supaya selesai beberapa menit. Untuk hasil sungguhan naikkan `N_TRAIN`, `EPOCHS` (idealnya dengan GPU). Bobot dibaca dari `pretrained/` (offline).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/classification/emot_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/classification/emot_test.csv"
TEXT_COL   = "text"
TARGET     = "label"
ID_COL     = 'id'
LANG       = "id"
SCORING    = "f1_macro"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "finetune_distilbert_id"
OUT = out_dir("text", "classification", MODEL_KEY)
set_seed(SEED)
MODEL_DIR  = "distilbert_id"       # folder di pretrained/
N_TRAIN, N_VALID, N_TEST = 800, 300, 500   # ukuran demo (CPU); naikkan saat lomba
EPOCHS, MAX_LEN, LR, BATCH = 1, 64, 3e-5, 16
SAVE_MODEL = False                          # True menyimpan bobot fine-tuned (±250-500 MB)
import torch; print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## 2. Data (subset berstrata)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from superdb.text import clean_text
df = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH) if TEST_PATH else None
le = LabelEncoder(); y_all = le.fit_transform(df[TARGET]); CLASSES = list(le.classes_)
idx = np.arange(len(df)); itr, iva = train_test_split(idx, train_size=min(N_TRAIN, len(df) - N_VALID), test_size=N_VALID, random_state=SEED, stratify=y_all)
txt = df[TEXT_COL].map(clean_text).tolist()
Xtr, ytr, Xva, yva = [txt[i] for i in itr], y_all[itr], [txt[i] for i in iva], y_all[iva]
print(len(Xtr), "latih /", len(Xva), "validasi | kelas:", CLASSES)

## 3. Pembanding: TF-IDF + Logistic Regression pada data yang sama

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
base = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LogisticRegression(C=5, max_iter=2000)).fit(Xtr, ytr)
f1_base = f1_score(yva, base.predict(Xva), average="macro"); print(f"F1 macro baseline TF-IDF: {f1_base:.4f}")

## 4. Fine-tuning

In [ ]:
from superdb.text_heavy import finetune_text_classifier, predict_proba_text
model, tok, hist = finetune_text_classifier(MODEL_DIR, Xtr, ytr, Xva, yva, len(CLASSES), epochs=EPOCHS, max_len=MAX_LEN, lr=LR, batch_size=BATCH, seed=SEED)
display(pd.DataFrame(hist))

## 5. Evaluasi

In [ ]:
proba = predict_proba_text(model, tok, Xva, MAX_LEN); pred = proba.argmax(1)
metrics = classification_metrics(yva, pred, proba, classes=np.arange(len(CLASSES))); metrics["f1_macro_tfidf_baseline"] = float(f1_base)
display(show_table(metrics))
print(f"F1 macro: fine-tune {metrics['f1_macro']:.4f} vs TF-IDF {f1_base:.4f} (data latih sama, {len(Xtr)} contoh, {EPOCHS} epoch)")
plot_confusion(yva, pred, labels=np.arange(len(CLASSES)), path=OUT / "confusion.png"); plt.show()

## 6. Prediksi test → submission

In [ ]:
if test is not None:
    t = test.head(N_TEST); pt = predict_proba_text(model, tok, t[TEXT_COL].map(clean_text).tolist(), MAX_LEN)
    sub = make_submission(t[ID_COL] if ID_COL else np.arange(len(t)), le.inverse_transform(pt.argmax(1)), ID_COL or "id", TARGET, OUT / "submission.csv"); display(sub.head(3))
if SAVE_MODEL:
    model.save_pretrained(OUT / "model"); tok.save_pretrained(OUT / "model")
save_json({"valid": metrics, "history": hist, "config": dict(N_TRAIN=N_TRAIN, EPOCHS=EPOCHS, MAX_LEN=MAX_LEN, LR=LR, BATCH=BATCH, model=MODEL_DIR)}, OUT / "metrics.json")
print("OK. hasil di", OUT)

## Langkah berikutnya
- Hasil demo kecil belum mewakili potensi model: naikkan `N_TRAIN` (ribuan), `EPOCHS` (3-4), `MAX_LEN` sesuai panjang teks.
- GPU: tidak perlu mengubah kode; `device` terdeteksi otomatis.
- Blend probabilitas fine-tuned dengan TF-IDF sering menambah skor.